# Q4)Writing Viterbi Algorithm for the Primer
## i) Viterbi algorithm to implment Nature Primer.


In [1]:
import numpy as np
import math

def log(x): return -math.inf if x == 0 else math.log(x)

trans_p = {
    'E': {'E': 0.9, '5': 0.1},
    '5': {'I': 1.0},
    'I': {'I': 0.9}
}
emit_p = {
    'E': {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25},
    '5': {'A': 0.05, 'C': 0.0, 'G': 0.95, 'T': 0.0},
    'I': {'A': 0.4, 'C': 0.1, 'G': 0.1, 'T': 0.4}
}

states = ['E', '5', 'I']
start_p = {'E': 1.0, '5': 0.0, 'I': 0.0}

We define the parameters of our **Hidden Markov Model (HMM)**:

---

### 🔹 `states`

- The three hidden states:
  - **Exon (E)**
  - **Donor site (5)**
  - **Intron (I)**

---

### 🔹 `start_p`

- The **starting probability** for each state.
- In this case, we **always start in an Exon (E)**, so:
  - `start_p = {'E': 1.0, '5': 0.0, 'I': 0.0}`

---

### 🔹 `trans_p`

- **Transition probabilities** between states.
- Example:
  - 90% chance of staying in **Exon**
  - 10% chance of transitioning from **Exon to Donor**

---

### 🔹 `emit_p`

- **Emission probabilities** for each nucleotide (`A`, `C`, `G`, `T`) in each state.
- For example:
  - In **Exon**, A might appear with a different probability than in **Intron**.

---


In [2]:
def get_log_prob_of_a_given_path(path: str, seq: str) -> float:
    if len(path) != len(seq):
        raise ValueError(f"Mismatch: path={len(path)}, seq={len(seq)}")

    prob = 0.0
    for i in range(len(seq)):
        x = path[i]
        y = seq[i]
        if i == 0:
            prob += log(start_p[x])
        else:
            prob += log(trans_p[path[i-1]][x])
        prob += log(emit_p[x][y])

    if path[-1] == 'I':
        prob += log(0.1)

    return round(prob, 2)

This function computes the **total log-probability** of a given path and observed sequence.

---

### 🔹 Input Validation

- It ensures that the **sequence** and **path** lengths match.

---

### 🔹 Log-Probability Calculation

For each position `i`, it adds:

- The **log of the start probability** (if `i = 0`) or the **log of the transition probability** (if `i > 0`)
- The **log of the emission probability** for the observed nucleotide at position `i`

---

### 🔹 Biological Adjustment

- An **additional penalty** (`log(0.1)`) is added **if the final state is `I`**, reflecting **biological assumptions** mentioned in the primer.

---

### 🔹 Final Output

- The result is **rounded to two decimal places** for readability.


In [3]:
p = "EEEEEEEEEEEEEEEEEE5IIIIIII"
s = "CTTCATGTGAAAGCAGACGTAAGTCA"
print("Log probability of given path:", get_log_prob_of_a_given_path(p, s))


Log probability of given path: -41.22


Here we test the function using:

p: A state path representing an initial exon stretch, a donor site, and an intron stretch.

s: A DNA sequence corresponding to the state path.

# ii) Getting the maximum likely path:

In [6]:
def viterbi(seq):
    V = [{}]
    path = {}

    for s in states:
        try:
            V[0][s] = log(start_p[s]) + log(emit_p[s][seq[0]])
        except KeyError:
            V[0][s] = -math.inf
        path[s] = [s]
    for t in range(1, len(seq)):
        V.append({})
        new_path = {}

        for curr in states:
            max_prob = -math.inf
            best_prev = None

            for prv in states:
                if trans_p.get(prv, {}).get(curr) is not None:
                    trans = log(trans_p[prv][curr])
                    emit = log(emit_p[curr].get(seq[t], 0))
                    prob = V[t-1][prv] + trans + emit

                    if prob > max_prob:
                        max_prob = prob
                        best_prev = prv

            V[t][curr] = max_prob
            if best_prev is not None:
                new_path[curr] = path[best_prev] + [curr]
            else:
                new_path[curr] = [curr]

        path = new_path
        n = len(seq) - 1
    final_prob, final_state = max((V[n][s], s) for s in states if s in path)

    if final_state not in path:
        raise ValueError("Final state not found in path")

    return round(final_prob, 2), ''.join(path[final_state])


This function implements the **Viterbi algorithm**, a dynamic programming method to find the most probable path of hidden states for a given sequence.

---

### 🔹 Data Structures Used

- `V` is a list of dictionaries. `V[t][s]` stores the **maximum log-probability** of any path that ends in state `s` at time `t`.

- `path` stores the **best sequence of states** leading to each state at time `t`.

---

### 🔹 Initialization Step (`t = 0`)

For each state `s`, we calculate:

- The log of the start probability: `start_p[s]`
- The log of the emission probability for the first character `seq[0]` from state `s`

If a key is missing, we assign `-inf` (i.e., that path is impossible).

We initialize the path to start with that state.

---

### 🔹 Main Loop (`t = 1` to `len(seq) - 1`)

For each current state `curr`, we consider all possible previous states `prv`.

We calculate the **log-probability** of:
- transitioning from `prv` to `curr`
- emitting the character `seq[t]` from `curr`

We choose the `prv` that gives the **highest probability path** to `curr`.

We update:
- `V[t][curr]` with the best log-probability for reaching `curr`
- `new_path[curr]` with the corresponding best path

After all states are processed at time `t`, `path` is updated to the best paths found so far.

---

### 🔹 Final Step

At the final time step `n`, we find the state `s` with the highest probability `V[n][s]`.

We retrieve the **best path** that ends in `final_state`.

---

### 🔹 Return Values

The function returns:
- The **maximum log-probability**, rounded to two decimal places.
- The **most probable path** as a string of states.

If no valid path leads to a final state, a `ValueError` is raised.


In [7]:
v_prob, v_path = viterbi(s)
print("Viterbi best log probability:", v_prob)
print("Most likely path:", v_path)


Viterbi best log probability: -38.68
Most likely path: EEEEEEEEEEEEEEEEEEEEEEEEEE


We run the viterbi() function on the sequence s:

v_prob holds the maximum log-probability of the best path.

v_path is the most probable sequence of hidden states that could have generated s.